In [1]:
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List
import json, uuid

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

root = Path.cwd() / 'agents026'
data_dir = root / 'data'
incidents_dir = data_dir / 'incidents'
exec_dir = data_dir / 'execution'
reports_dir = data_dir / 'reports'
audit_dir = data_dir / 'audit'
prevention_dir = data_dir / 'prevention'

OPERATOR = 'prasenjit.roychoudhury'
AUDIT_LOG_PATH = audit_dir / 'audit_log.jsonl'

# Load artifacts
incidents = json.load(open(incidents_dir / 'incident_candidates.json', encoding='utf-8'))
incident_map = {x['incident_id']: x for x in incidents}
metrics_df = pd.read_csv(data_dir / 'metrics.csv', parse_dates=['timestamp'])

workflow_map = {}
for fp in sorted(exec_dir.glob('*_workflow.json')):
    b = json.load(open(fp, encoding='utf-8'))
    workflow_map[b['incident_id']] = b

def audit_event(event_type, incident_id, payload):
    rec = {'audit_id': f'aud-{uuid.uuid4().hex[:12]}',
           'timestamp_utc': datetime.now(timezone.utc).isoformat(),
           'event_type': event_type, 'incident_id': incident_id,
           'operator': OPERATOR, 'payload': payload}
    with open(AUDIT_LOG_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(rec, default=str) + '\n')
    return rec

print(f'Loaded: {len(incidents)} incidents | {len(workflow_map)} workflows | reports: {len(list(reports_dir.glob("*_report.md")))}')

Loaded: 14 incidents | 2 workflows | reports: 2


In [2]:
incident_dd = widgets.Dropdown(
    options=[(f"{x['incident_id']} — {','.join(x['services'])} ({x['anomaly_type']})", x['incident_id'])
             for x in incidents],
    value='inc-014',
    description='Incident:',
    layout=widgets.Layout(width='480px'),
)
incident_out = widgets.Output()

def render_incident(change=None):
    iid = incident_dd.value
    inc = incident_map[iid]
    with incident_out:
        clear_output()
        ms = inc.get('metric_summary', {})
        display(Markdown(
            f"### {iid} — `{','.join(inc['services'])}`\n"
            f"**Anomaly:** {inc['anomaly_type']} | **Window:** {inc['start_time']} → {inc['end_time']}\n\n"
            f"| cpu_max | latency_p95_max | error_rate_max |\n|---|---|---|\n"
            f"| {ms.get('cpu_max',0):.1f}% | {ms.get('latency_p95_max',0):.0f}ms | {ms.get('error_rate_max',0):.4f} |"
        ))
        if inc.get('change_refs'):
            display(Markdown('**Recent changes:**\n' + '\n'.join(f'- {c}' for c in inc['change_refs'][:4])))

        svc = inc['services'][0]
        s, e = pd.Timestamp(inc['start_time']), pd.Timestamp(inc['end_time'])
        sdf = metrics_df[(metrics_df.service == svc) &
                         (metrics_df.timestamp >= s - pd.Timedelta(minutes=40)) &
                         (metrics_df.timestamp <= e + pd.Timedelta(minutes=40))]
        fig, axes = plt.subplots(1, 3, figsize=(14, 3))
        for ax, m in zip(axes, ['cpu_utilization', 'latency_p95_ms', 'error_rate']):
            ax.plot(sdf.timestamp, sdf[m], lw=1.2)
            ax.axvspan(s, e, color='red', alpha=0.15)
            ax.set_title(m, fontsize=9)
            ax.tick_params(axis='x', rotation=45, labelsize=7)
        plt.tight_layout(); plt.show()

incident_dd.observe(render_incident, names='value')
render_incident()
tab_incidents = widgets.VBox([incident_dd, incident_out])
print('Tab 1 ready')

Tab 1 ready


In [3]:
rca_out = widgets.Output()

def render_rca(change=None):
    iid = incident_dd.value
    with rca_out:
        clear_output()
        if iid not in workflow_map:
            display(Markdown(f'*No workflow has been run for `{iid}` yet — run it via the Hour 5–6 pipeline.*'))
            return
        b = workflow_map[iid]
        rca = b['rca_result']
        display(Markdown(
            f"### 🔍 RCA — {iid}\n"
            f"**Root cause:** {rca['root_cause_hypothesis']}\n\n"
            f"**Trigger:** `{rca.get('probable_trigger')}` | **Confidence:** {rca['confidence']:.0%}\n\n"
            f"**Evidence:**\n" + '\n'.join(f'- {e}' for e in rca['evidence'])
        ))
        actions = pd.DataFrame(b['action_plan']['actions'])
        display(Markdown('### ⚙️ Action plan & execution status'))
        display(actions[['action_type', 'target', 'requires_approval', 'status']]
                .style.map(lambda v: 'color: green; font-weight: bold' if v == 'executed'
                           else ('color: red' if v == 'rejected'
                           else ('color: orange' if v == 'skipped' else '')), subset=['status']))

incident_dd.observe(render_rca, names='value')
render_rca()
tab_rca = widgets.VBox([rca_out])
print('Tab 2 ready')

Tab 2 ready


In [4]:
EXECUTORS = {
    'rollback_deploy': lambda a: {'status': 'executed', 'message': f"Simulated rollback of {a['target']} to {a.get('parameters',{}).get('version','previous')}"},
    'rollback_config': lambda a: {'status': 'executed', 'message': f"Simulated config rollback for {a['target']}"},
    'disable_feature_flag': lambda a: {'status': 'executed', 'message': f"Simulated flag disable on {a['target']}"},
    'restart_service': lambda a: {'status': 'executed', 'message': f"Simulated restart of {a['target']}"},
    'scale_service': lambda a: {'status': 'executed', 'message': f"Simulated scale of {a['target']}"},
}

queue_container = widgets.VBox([])
queue_refresh_btn = widgets.Button(description='🔄 Refresh queue', button_style='info')

def build_queue(change=None):
    iid = incident_dd.value
    rows = []
    if iid in workflow_map:
        pending = [a for a in workflow_map[iid]['execution_result']['skipped_actions'] if a['status'] == 'skipped']
        for action in pending:
            header = widgets.HTML(
                f"<b>{action['action_type']}</b> → <code>{action['target']}</code><br>"
                f"<i>{action['description']}</i><br><b>Impact:</b> {action.get('expected_impact','-')}"
            )
            ok = widgets.Button(description='✅ Approve', button_style='success')
            no = widgets.Button(description='❌ Reject', button_style='danger')
            lbl = widgets.HTML("<b style='color:#b58900'>⏳ PENDING</b>")

            def handler(btn, action=action, ok=ok, no=no, lbl=lbl, iid=iid):
                decision = 'approved' if btn is ok else 'rejected'
                audit_event('human_decision', iid, {'action_id': action['action_id'], 'decision': decision})
                if decision == 'approved':
                    res = EXECUTORS.get(action['action_type'], lambda a: {'status':'failed','message':'no executor'})(action)
                    action['status'] = res['status']
                    audit_event('action_executed', iid, {'action_id': action['action_id'], 'result': res})
                    lbl.value = f"<b style='color:#2aa198'>✅ EXECUTED</b> — {res['message']}"
                else:
                    action['status'] = 'rejected'
                    audit_event('action_rejected', iid, {'action_id': action['action_id']})
                    lbl.value = "<b style='color:#dc322f'>❌ REJECTED</b>"
                # persist workflow update
                b = workflow_map[iid]
                ex = [a for a in b['execution_result']['skipped_actions'] if a['status'] == 'executed']
                b['execution_result']['executed_actions'].extend(ex)
                b['execution_result']['skipped_actions'] = [a for a in b['execution_result']['skipped_actions'] if a['status'] != 'executed']
                json.dump(b, open(exec_dir / f'{iid}_workflow.json', 'w', encoding='utf-8'), default=str, indent=2)
                ok.disabled = no.disabled = True

            ok.on_click(handler); no.on_click(handler)
            rows.append(widgets.VBox([header, widgets.HBox([ok, no, lbl])],
                        layout=widgets.Layout(border='1px solid #ccc', padding='8px', margin='4px 0')))
    queue_container.children = rows if rows else [widgets.HTML('<i>No actions pending approval for this incident ✅</i>')]

queue_refresh_btn.on_click(build_queue)
incident_dd.observe(build_queue, names='value')
build_queue()
tab_queue = widgets.VBox([queue_refresh_btn, queue_container])
print('Tab 3 ready')

Tab 3 ready


In [5]:
gov_out = widgets.Output()

def render_gov(_=None):
    with gov_out:
        clear_output()
        prev_path = prevention_dir / 'prev-001_prevention.json'
        if prev_path.exists():
            p = json.load(open(prev_path, encoding='utf-8'))
            display(Markdown(
                f"### 🔮 Predictive Prevention (UC-5)\n"
                f"`{p['service']}` — breach forecast **{p['lead_time_minutes']} min ahead**, "
                f"preventive `{p['preventive_action']['action_type']}` executed → **{p['outcome'].replace('_',' ')}**"
            ))
        if AUDIT_LOG_PATH.exists():
            adf = pd.DataFrame([json.loads(l) for l in open(AUDIT_LOG_PATH, encoding='utf-8')])
            display(Markdown(f'### 📋 Audit Trail — {len(adf)} records (append-only)'))
            display(adf[['timestamp_utc', 'event_type', 'incident_id', 'operator']].tail(12))

gov_refresh = widgets.Button(description='🔄 Refresh audit', button_style='info')
gov_refresh.on_click(render_gov)
render_gov()
tab_gov = widgets.VBox([gov_refresh, gov_out])
print('Tab 4 ready')

Tab 4 ready


In [6]:
console = widgets.Tab(children=[tab_incidents, tab_rca, tab_queue, tab_gov])
console.set_title(0, '🚨 Incidents')
console.set_title(1, '🔍 RCA & Actions')
console.set_title(2, '🛡️ Approval Queue')
console.set_title(3, '📋 Governance')

display(Markdown('# AGENTS026 — SRE Operator Console\n*Autonomous Incident Diagnosis & Resolution · AMD MI300X*'))
display(console)

# AGENTS026 — SRE Operator Console
*Autonomous Incident Diagnosis & Resolution · AMD MI300X*